# 🏆 Benchmark 360: Inteligencia Competitiva Automatizada
## Mercado de Internet Fijo — Ecuador 2026

---
**Cliente:** Megadatos S.A. / Netlife
**Equipo:** FerQode Dev Team
**Fecha:** Abril 2026

---

### El Problema en Números
> En Ecuador existen **+1,400 ISPs** compitiendo en un mercado de precio altamente
> volátil. El equipo de Pricing de Netlife tardaba **semanas** en detectar cambios
> en la oferta de competidores clave como Claro, Xtrim o CNT.

### La Solución
Un **Data Pipeline Enterprise** que ejecuta diariamente y entrega en menos de
**24 horas** el 100% de la oferta comercial vigente de los principales ISPs
competidores, incluyendo precios en imágenes y términos y condiciones completos.

## 🏗️ Arquitectura del Pipeline

```
INTERNET (páginas web de 8 ISPs)
          │
          ▼
┌─────────────────────────────────────────────────────┐
│              CAPA 1: EXTRACCIÓN                     │
│  ISPStrategy ──► BaseISPScraper ──► CookieHandler   │
│  TCHTMLScraper ──► Términos y Condiciones (sin IA)  │
└─────────────────────────────────────────────────────┘
          │                    │
     Texto HTML (37KB)   7 Tiles PNG
          │                    │
          ▼                    ▼
┌─────────────────────────────────────────────────────┐
│              CAPA 2: INTELIGENCIA ARTIFICIAL        │
│  LLMProcessor            VisionProcessor            │
│  gemini-2.5-flash   →    gemini-3.1-pro (Tier 1)   │
│  flash-lite         →    pixtral-large  (Tier 2)   │
│  gpt-4o-mini        →    flash-lite     (Tier 3)   │
│                     →    gpt-4o-mini    (Tier 4)   │
│  + Guardrails (Anti Prompt Injection)               │
└─────────────────────────────────────────────────────┘
          │
          ▼
┌─────────────────────────────────────────────────────┐
│              CAPA 3: VALIDACIÓN Y STORAGE           │
│  Pydantic V2 ──► PlanNormalizer ──► Parquet         │
│  (30 campos)    (snake_case, IVA, dedup)  (por fecha)│
└─────────────────────────────────────────────────────┘
```
**Stack:** Python 3.12 · uv · Playwright · google-genai · Mistral · Pydantic V2 · Pandas · PyArrow

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from datetime import datetime
import json, ast

plt.rcParams.update({
    "figure.figsize": (14, 7),
    "figure.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
})

ISP_COLORS = {
    "Netlife":  "#E31837",
    "Claro":    "#DA291C",
    "Xtrim":    "#00A859",
    "CNT":      "#005BAA",
    "Ecuanet":  "#FF6B00",
    "Fibramax": "#8B00FF",
    "Alfanet":  "#00B4D8",
    "Puntonet": "#2D6A4F",
}

Path("data/output/charts").mkdir(parents=True, exist_ok=True)
print("Libraries loaded")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Libraries loaded
Analysis date: 2026-04-18 19:46


In [2]:
PARQUET_PATH = Path("data/output/benchmark_industria.parquet")

def safe_parse_dict(v):
    if pd.isna(v) or str(v) in ("", "{}", "None"): return {}
    try: return json.loads(v)
    except: 
        try: return ast.literal_eval(str(v))
        except: return {}

def safe_parse_list(v):
    if pd.isna(v) or str(v) in ("", "[]", "None"): return []
    try:
        r = ast.literal_eval(str(v))
        return r if isinstance(r, list) else []
    except: return []

df = pd.read_parquet(PARQUET_PATH)
df["pys_adicionales_detalle"] = df["pys_adicionales_detalle"].apply(safe_parse_dict)
df["sectores"] = df["sectores"].apply(safe_parse_list)
df["fecha"] = pd.to_datetime(df["fecha"])
df["precio_plan"] = pd.to_numeric(df["precio_plan"], errors="coerce")
df["velocidad_download_mbps"] = pd.to_numeric(df["velocidad_download_mbps"], errors="coerce")

print("=" * 60)
print("  BENCHMARK 360 — Dataset Cargado")
print("=" * 60)
print(f"  Registros:          {len(df)}")
print(f"  ISPs:               {df['marca'].nunique()} ({', '.join(sorted(df['marca'].unique()))})")
print(f"  Columnas:           {len(df.columns)}")
print(f"  Fecha extraccion:   {df['fecha'].max().strftime('%Y-%m-%d %H:%M')}")
print(f"  Rango precios:      ${df['precio_plan'].min():.2f} - ${df['precio_plan'].max():.2f}")
print(f"  Rango velocidades:  {df['velocidad_download_mbps'].min():.0f} - {df['velocidad_download_mbps'].max():.0f} Mbps")
print("=" * 60)

df[["marca","nombre_plan","velocidad_download_mbps","precio_plan","tecnologia","pys_adicionales"]].head(10)

  BENCHMARK 360 — Dataset Cargado
  Registros:          29
  ISPs:               8 (Alfanet, CNT, Claro, Ecuanet, Fibramax, Netlife, Puntonet, Xtrim)
  Columnas:           30
  Fecha extraccion:   2026-04-18 17:37
  Rango precios:      $15.22 - $86.96
  Rango velocidades:  100 - 2000 Mbps


,marca,nombre_plan,velocidad_download_mbps,precio_plan,tecnologia,pys_adicionales
0,Claro,Internet + Telefonía 550Mbps,550.0,17.39,NaN,1
1,Claro,Internet + Telefonía 850Mbps,850.0,21.74,NaN,1
2,Xtrim,Plan Essential 2,200.0,24.35,fibra_optica,0
3,Xtrim,Plan Elite Pro,800.0,24.35,fibra_optica,1
4,Xtrim,Plan Supreme Pro,1000.0,30.24,fibra_optica,3
5,Xtrim,Plan Prime,1000.0,32.14,fibra_optica,2
6,Xtrim,1000mega Plan Prime,1000.0,32.14,fibra_optica,0
7,Xtrim,800 megas Plan Elite Pro,800.0,21.17,NaN,2
8,Netlife,Plan Netlife 750 Mbps,750.0,22.39,fttp,2
9,Fibramax,Plan 700 Megas,700.0,15.22,fibra_optica,0


## ✅ Sección 4: Validación de Calidad de Datos

Antes de cualquier análisis auditamos completitud y consistencia del dataset.
El pipeline usa **Pydantic V2** para garantizar que cada campo cumple las reglas
de negocio definidas en el schema (30 campos, tipos estrictos, rangos válidos).

**Métricas clave:**
- Completitud por columna (% valores no-nulos)
- Consistencia del cálculo de descuentos (precio_plan → precio_plan_descuento)
- Unicidad de planes por ISP (deduplicación por nombre)

In [3]:
columnas_criticas = [
    "marca","nombre_plan","velocidad_download_mbps",
    "precio_plan","tecnologia","terminos_condiciones",
    "pys_adicionales","descuento",
]
completitud = (df.notna().mean() * 100).sort_values(ascending=False).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

comp_criticas = completitud[columnas_criticas].sort_values()
colors_bar = ["#E31837" if v < 80 else "#00A859" for v in comp_criticas]
comp_criticas.plot(kind="barh", ax=axes[0], color=colors_bar, edgecolor="white")
axes[0].set_title("Completitud — Campos Criticos de Negocio", fontweight="bold")
axes[0].set_xlabel("% de registros con dato")
axes[0].axvline(x=80, color="red", linestyle="--", alpha=0.7, label="Meta >80%")
axes[0].legend()
for i, v in enumerate(comp_criticas):
    axes[0].text(v + 0.5, i, f"{v:.0f}%", va="center", fontsize=9)

registros_isp = df["marca"].value_counts()
registros_isp.plot(
    kind="bar", ax=axes[1],
    color=[ISP_COLORS.get(isp, "#666") for isp in registros_isp.index],
    edgecolor="white",
)
axes[1].set_title("Planes Extraidos por ISP", fontweight="bold")
axes[1].tick_params(axis="x", rotation=45)
for i, v in enumerate(registros_isp):
    axes[1].text(i, v + 0.1, str(v), ha="center", fontsize=10, fontweight="bold")

plt.suptitle("Benchmark 360 — Auditoria de Calidad del Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("data/output/charts/01_calidad_datos.png", bbox_inches="tight", dpi=150)
plt.show()

score = completitud[columnas_criticas].mean()
print(f"Score de Completitud Global: {score:.1f}%")
print(f"Criterio >80%: {'CUMPLIDO' if score >= 80 else 'EN PROGRESO'}")

Score de Completitud Global: 82.8%
Criterio >80%: CUMPLIDO


## 💰 Sección 5: Análisis Competitivo de Precios

El corazón del caso de negocio. El equipo de Pricing de Netlife necesita saber:

1. **¿Quién es el más barato?** → Amenaza directa al volumen
2. **¿Quién da más por el mismo precio?** → Amenaza a la percepción de valor
3. **¿Quién usa descuentos agresivos?** → Estrategia de captación temporal

> **Insight para Netlife:** Un descuento del 30% en los primeros 3 meses
> equivale a bajar el precio real un 7.5% anual. El pipeline detecta esto automáticamente.

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Gráfico 1: Precio promedio por ISP
precio_por_isp = df.groupby("marca")["precio_plan"].agg(["mean","min","max"]).sort_values("mean")
ax1 = axes[0, 0]
bars = ax1.barh(
    precio_por_isp.index, precio_por_isp["mean"],
    color=[ISP_COLORS.get(isp, "#999") for isp in precio_por_isp.index],
    edgecolor="white", height=0.6,
)
ax1.errorbar(
    precio_por_isp["mean"], range(len(precio_por_isp)),
    xerr=[precio_por_isp["mean"]-precio_por_isp["min"], precio_por_isp["max"]-precio_por_isp["mean"]],
    fmt="none", color="#333", capsize=4, linewidth=2,
)
ax1.set_title("Precio Promedio por ISP (sin IVA)\n[barra=promedio, linea=rango]")
ax1.set_xlabel("Precio mensual USD (sin IVA)")
for bar, val in zip(bars, precio_por_isp["mean"]):
    ax1.text(val+0.2, bar.get_y()+bar.get_height()/2, f"${val:.2f}", va="center", fontsize=9, fontweight="bold")

# Gráfico 2: Violin plot
ax2 = axes[0, 1]
isps_ord = df.groupby("marca")["precio_plan"].median().sort_values().index.tolist()
data_violin = [df[df["marca"]==isp]["precio_plan"].dropna().values for isp in isps_ord]
parts = ax2.violinplot(data_violin, positions=range(len(isps_ord)), showmedians=True)
for pc, isp in zip(parts["bodies"], isps_ord):
    pc.set_facecolor(ISP_COLORS.get(isp, "#999"))
    pc.set_alpha(0.7)
ax2.set_xticks(range(len(isps_ord)))
ax2.set_xticklabels(isps_ord, rotation=45, ha="right")
ax2.set_title("Distribucion de Precios por ISP")
ax2.set_ylabel("Precio mensual USD (sin IVA)")

# Gráfico 3: Heatmap
ax3 = axes[1, 0]
df["velocidad_bin"] = pd.cut(
    df["velocidad_download_mbps"],
    bins=[0,200,400,600,800,1000,float("inf")],
    labels=["<=200","201-400","401-600","601-800","801-1000",">1000"],
)
pivot = df.pivot_table(values="precio_plan", index="marca", columns="velocidad_bin", aggfunc="mean")
sns.heatmap(pivot, ax=ax3, cmap="RdYlGn_r", annot=True, fmt=".1f",
            cbar_kws={"label":"Precio USD"}, linewidths=0.5)
ax3.set_title("Heatmap: Precio por ISP y Velocidad\n[verde=barato]")
ax3.set_xlabel("Rango velocidad (Mbps)")

# Gráfico 4: Descuentos
ax4 = axes[1, 1]
df_d = df[df["descuento"].notna() & (df["descuento"]>0)].copy()
df_d["descuento_pct"] = df_d["descuento"]*100
if len(df_d) > 0:
    desc_isp = df_d.groupby("marca").agg(
        desc_prom=("descuento_pct","mean"), n_planes=("nombre_plan","count")
    ).sort_values("desc_prom", ascending=False)
    bars4 = ax4.bar(desc_isp.index, desc_isp["desc_prom"],
                    color=[ISP_COLORS.get(i,"#999") for i in desc_isp.index], edgecolor="white")
    ax4.set_title("Descuentos Promocionales por ISP")
    ax4.set_ylabel("Descuento promedio (%)")
    ax4.tick_params(axis="x", rotation=45)
    for bar, (idx, row) in zip(bars4, desc_isp.iterrows()):
        ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f"{row['desc_prom']:.1f}%", ha="center", fontsize=9)
else:
    ax4.text(0.5, 0.5, "Sin datos de descuento", ha="center", va="center", transform=ax4.transAxes)
    ax4.set_title("Descuentos Promocionales")

plt.suptitle("Benchmark 360 — Analisis Competitivo de Precios", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("data/output/charts/02_analisis_precios.png", bbox_inches="tight", dpi=150)
plt.show()

In [5]:
df_valor = df[df["precio_plan"].notna() & df["velocidad_download_mbps"].notna() & (df["precio_plan"]>0)].copy()
df_valor["mbps_por_dolar"] = (df_valor["velocidad_download_mbps"] / df_valor["precio_plan"]).round(2)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Scatter
ax1 = axes[0]
for isp, grp in df_valor.groupby("marca"):
    ax1.scatter(grp["precio_plan"], grp["velocidad_download_mbps"],
                c=ISP_COLORS.get(isp,"#999"), s=120, alpha=0.8, label=isp,
                edgecolors="white", linewidth=1.5, zorder=3)
    for _, row in grp.iterrows():
        nm = str(row["nombre_plan"])[:14]
        ax1.annotate(nm, (row["precio_plan"], row["velocidad_download_mbps"]),
                     textcoords="offset points", xytext=(5,3), fontsize=6.5, alpha=0.8)
avg_p = df_valor["precio_plan"].mean()
avg_v = df_valor["velocidad_download_mbps"].mean()
ax1.axvline(avg_p, color="gray", linestyle="--", alpha=0.5, label=f"Precio prom ${avg_p:.1f}")
ax1.axhline(avg_v, color="gray", linestyle=":", alpha=0.5, label=f"Vel prom {avg_v:.0f} Mbps")
ax1.text(df_valor["precio_plan"].min()+1, df_valor["velocidad_download_mbps"].max()*0.93,
         "MEJOR VALOR\n(Rapido y Barato)", fontsize=8, color="green", alpha=0.8)
ax1.text(df_valor["precio_plan"].max()*0.70, df_valor["velocidad_download_mbps"].min()+50,
         "PEOR VALOR\n(Lento y Caro)", fontsize=8, color="red", alpha=0.8)
ax1.set_title("Mapa Competitivo: Velocidad vs Precio")
ax1.set_xlabel("Precio mensual USD (sin IVA)")
ax1.set_ylabel("Velocidad de descarga (Mbps)")
ax1.legend(loc="lower right", fontsize=8, ncol=2)

# Bar: Mbps/$
ax2 = axes[1]
indice = df_valor.groupby("marca")["mbps_por_dolar"].mean().sort_values(ascending=False)
mkt_avg = indice.mean()
bars = ax2.bar(indice.index, indice.values,
               color=[ISP_COLORS.get(i,"#999") for i in indice.index],
               edgecolor="white", width=0.6)
ax2.axhline(mkt_avg, color="black", linestyle="--", alpha=0.6, label=f"Promedio mercado: {mkt_avg:.1f} Mbps/$")
ax2.set_title("Indice de Valor Competitivo\n[Mbps por dolar — mayor es mejor]")
ax2.set_ylabel("Mbps por dolar (USD)")
ax2.tick_params(axis="x", rotation=45)
ax2.legend()
for bar, val in zip(bars, indice.values):
    color = "green" if val > mkt_avg else "red"
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
             f"{val:.1f}", ha="center", fontsize=11, fontweight="bold", color=color)

plt.suptitle("Benchmark 360 — Indice de Valor (Mbps/$)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig("data/output/charts/03_indice_valor.png", bbox_inches="tight", dpi=150)
plt.show()

print("\nRANKING: Mejor Valor (Mbps por Dolar)")
print("-" * 45)
for rank, (isp, val) in enumerate(indice.items(), 1):
    emoji = ["1.", "2.", "3."][rank-1] if rank <= 3 else f"{rank}."
    diff = val - mkt_avg
    print(f"  {emoji} {isp:<12} {val:>6.1f} Mbps/$  ({'▲' if diff>0 else '▼'}{abs(diff):.1f} vs promedio)")


RANKING: Mejor Valor (Mbps por Dolar)
---------------------------------------------
  1. Fibramax       46.0 Mbps/$  (▲22.1 vs promedio)
  2. Claro          35.2 Mbps/$  (▲11.3 vs promedio)
  3. Xtrim          28.6 Mbps/$  (▲4.7 vs promedio)
  4. Netlife        25.8 Mbps/$  (▲2.0 vs promedio)
  5. Alfanet        16.0 Mbps/$  (▼7.8 vs promedio)
  6. Ecuanet        13.9 Mbps/$  (▼9.9 vs promedio)
  7. Puntonet       13.7 Mbps/$  (▼10.1 vs promedio)
  8. CNT            11.5 Mbps/$  (▼12.3 vs promedio)


In [6]:
todos_servicios = []
for _, row in df.iterrows():
    detalle = row.get("pys_adicionales_detalle", {})
    if isinstance(detalle, dict):
        for svc, info in detalle.items():
            categoria = info.get("categoria","otros") if isinstance(info,dict) else "otros"
            meses = info.get("meses") if isinstance(info,dict) else None
            todos_servicios.append({"isp":row["marca"],"servicio":svc,"categoria":categoria,"meses":meses})

df_svc = pd.DataFrame(todos_servicios)

if len(df_svc) == 0:
    print("Sin datos de servicios adicionales en el dataset.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    top_svc = df_svc["servicio"].value_counts().head(8)
    top_svc.plot(kind="barh", ax=axes[0], color="#E31837", edgecolor="white")
    axes[0].set_title("Top Servicios Adicionales mas Ofrecidos")
    axes[0].set_xlabel("Numero de planes que lo incluyen")

    cat_counts = df_svc["categoria"].value_counts()
    axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct="%1.1f%%",
                colors=plt.cm.Set3.colors[:len(cat_counts)], startangle=90)
    axes[1].set_title("Distribucion por Categoria")

    plt.suptitle("Benchmark 360 — Servicios Adicionales (OTT/Streaming)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig("data/output/charts/04_servicios_ott.png", bbox_inches="tight", dpi=150)
    plt.show()

    print("\nPresencia de Servicios por ISP:")
    pivot_svc = df_svc.pivot_table(values="meses", index="isp", columns="servicio", aggfunc="first").notna()
    print(pivot_svc.replace({True:"Incluido", False:""}).to_string())


Presencia de Servicios por ISP:


servicio disney_plus kaspersky       max paramount_plus
isp                                                    
Claro       Incluido                                   
Puntonet              Incluido                         
Xtrim       Incluido            Incluido       Incluido


In [7]:
resumen = df.groupby("marca").agg(
    planes=("nombre_plan","count"),
    precio_min=("precio_plan","min"),
    precio_max=("precio_plan","max"),
    precio_prom=("precio_plan","mean"),
    vel_max=("velocidad_download_mbps","max"),
    vel_prom=("velocidad_download_mbps","mean"),
    con_descuento=("descuento", lambda x: (x>0).sum()),
    desc_max=("descuento", lambda x: round(x.max()*100,1) if x.notna().any() else 0),
    ott=("pys_adicionales","sum"),
    con_tc=("terminos_condiciones", lambda x: x.notna().sum()),
).round(2)

print("=" * 85)
print("  RESUMEN EJECUTIVO — BENCHMARK COMPETITIVO ISPs ECUADOR")
print(f"  Extraccion: {df['fecha'].max().strftime('%Y-%m-%d %H:%M')}")
print("=" * 85)

disp = resumen.copy()
for c in ["precio_min","precio_max","precio_prom"]:
    disp[c] = disp[c].apply(lambda x: f"${x:.2f}")
disp["vel_max"] = disp["vel_max"].apply(lambda x: f"{x:.0f} Mbps")
disp["desc_max"] = disp["desc_max"].apply(lambda x: f"{x:.1f}%")
disp.columns = ["Planes","Min","Max","Prom","Vel Max","Vel Prom","c/Dscto","Dscto Max","OTT","Con T&C"]
print(disp.to_string())

print("\nALERTAS COMPETITIVAS DETECTADAS:")
precio_netlife = df[df["marca"]=="Netlife"]["precio_plan"].mean() if "Netlife" in df["marca"].values else None
for isp, row in resumen.iterrows():
    if isp == "Netlife": continue
    if precio_netlife and row["precio_prom"] < precio_netlife * 0.85:
        diff = (precio_netlife - row["precio_prom"])/precio_netlife*100
        print(f"  ALERTA: {isp} tiene precios {diff:.1f}% mas bajos que Netlife")
    if row["desc_max"] > 20:
        print(f"  ALERTA: {isp} ofrece descuentos de hasta {row['desc_max']:.1f}% — campaña agresiva")
    if row["vel_max"] > 800:
        print(f"  INFO:   {isp} ofrece hasta {row['vel_max']} — presion en segmento premium")

  RESUMEN EJECUTIVO — BENCHMARK COMPETITIVO ISPs ECUADOR
  Extraccion: 2026-04-18 17:37
          Planes     Min     Max    Prom    Vel Max  Vel Prom  c/Dscto Dscto Max  OTT  Con T&C
marca                                                                                         
Alfanet        2  $17.39  $24.35  $20.87   500 Mbps    350.00        0      0.0%    0        0
CNT            3  $18.26  $34.78  $26.16   600 Mbps    333.33        0      0.0%    0        3
Claro          3  $17.39  $28.70  $22.61  1000 Mbps    800.00        0      0.0%    3        0
Ecuanet        2  $19.13  $28.70  $23.92   500 Mbps    350.00        2     22.7%    0        0
Fibramax       1  $15.22  $15.22  $15.22   700 Mbps    700.00        0      0.0%    0        0
Netlife        7  $22.39  $86.96  $49.68  2000 Mbps   1207.14        7     50.0%   10        4
Puntonet       2  $20.00  $28.70  $24.35   500 Mbps    350.00        0      0.0%    2        0
Xtrim          9  $21.17  $32.14  $26.65  1000 Mbps    77

## 🎯 Sección 9: Conclusiones y Recomendaciones para Netlife

### Hallazgos Principales

#### 1. 💰 Presión en el Segmento de Precio Bajo (<$25/mes)
Xtrim, Fibramax y Alfanet concentran su oferta en velocidades altas (500-1000 Mbps)
a precios agresivos (<$25/mes sin IVA), capturando el segmento price-sensitive
con una propuesta Mbps/$ superior al promedio del mercado.

**Recomendación:** Netlife debe evaluar un plan de entrada $15-20 con mínimo
200 Mbps para no perder usuarios de primer acceso frente a Fibramax.

#### 2. 🎬 El Streaming como Diferenciador Clave
Xtrim y Claro incluyen Disney+, Max y otros OTT como beneficio temporal.
Esta estrategia **aumenta la percepción de valor sin reducir el precio base**.

**Recomendación:** Fortalecer Netlife Defense (Kaspersky) y añadir al menos
1 servicio OTT por plan premium como diferenciador frente a CNT.

#### 3. ⚡ Oportunidad en el Segmento Premium (>500 Mbps)
Ningún ISP se posiciona claramente como "el premium de Ecuador".

**Recomendación:** Oportunidad para Netlife de ocupar posicionamiento
premium con SLA garantizado + soporte 24/7 + OTT bundle.

---

### Valor Entregado por Benchmark 360

| Antes del Pipeline | Con Benchmark 360 |
|---|---|
| Detección de cambios: 2-3 semanas | Detección: < 24 horas |
| Datos en Excel manual | Parquet analítico listo para BI |
| Solo precios de texto | Precios en imágenes incluidos |
| Sin histórico | Trazabilidad completa por fecha |
| 1 analista 8h/semana | 0 horas humanas + pipeline diario |

---

### Próximos Pasos

1. **Ampliar a 50+ ISPs** usando el mismo pipeline dinámico
2. **Dashboard en tiempo real** conectado al Parquet (Tableau/Power BI)
3. **Alertas automáticas** vía Slack/Teams cuando un competidor cambia precios >5%
4. **Análisis de series de tiempo** para detectar patrones de temporada